In [1]:
import os
import re
from nltk.stem import PorterStemmer

dir_name = '/Users/niyaznabi/Desktop/IR Project/ft911'  
 

# Load stop words
with open("/Users/niyaznabi/Desktop/IR Project/stopwordlist.txt", 'r') as f:
    stop_words_tokenize = f.read()
    stop_words_tokenize = stop_words_tokenize.replace("    ", "")
    stop_words_tokenize = stop_words_tokenize.split("\n")

# Initialize dictionaries
dict_wording = {}
dict_filing = {}
word_ids = 1
doc_ids = 1

# Output containers for token and document information
output_tokens = []
docs_output = []

# Function to add a document to the file dictionary
def Documents_Add(doc_name):
    global doc_ids
    if doc_name not in dict_filing:
        dict_filing[doc_name] = doc_ids
        doc_ids += 1
    return dict_filing[doc_name]

# Function to add a word to the word dictionary (with stemming)
def Words_add(word):
    global word_ids
    ps = PorterStemmer()
    word_stemmed = ps.stem(word)
    if word_stemmed not in dict_wording:
        dict_wording[word_stemmed] = word_ids
        word_ids += 1
    return dict_wording[word_stemmed]

# Process files in the directory
for file_name in os.listdir(dir_name):
    with open(os.path.join(dir_name, file_name), 'r') as f:
        content = f.read().split("</DOC>")
        
        for i in range(len(content)):
            # Searching for the DOCNO tag
            doc_name = content[i][content[i].find("<DOCNO>")+7:content[i].find("</DOCNO>")] 
            if doc_name:
                doc_id = Documents_Add(doc_name)
                docs_output.append(f"{doc_name}\t{doc_id}")  # Formatting doc name with ID

            # Searching for TEXT Tag and converting text into lowercase 
            text_start = content[i].find("<TEXT>") + 6
            text_end = content[i].find("</TEXT>")
            
            if text_start > 6 and text_end != -1:  # Ensure <TEXT> exists
                text = content[i][text_start:text_end].lower()
                text = text.replace('\n', ' ')            

                for words in text.split(' '):
                    # Splitting at non-alphanumeric characters
                    for word in re.split('[^a-zA-Z0-9]', words):
                        # If word starts with 'ft911', it should be included
                        if word.startswith("ft911"):
                            word_id = Words_add(word)
                            output_tokens.append(f"{word}\t{word_id}")
                        # Ignoring words with numbers, eliminating stop words
                        elif word and (word not in stop_words_tokenize) and word.isalpha():
                            word_id = Words_add(word)
                            output_tokens.append(f"{word}\t{word_id}")

# Writing the output to file in the desired format
with open("parser_output.txt", 'w') as f: 
    # Writing tokens and their IDs in required format
    for token in output_tokens:
        f.write(f"{token}\n")
    
    f.write("\n------------------------------------------------------\n\n")
    
    # Writing document names and their IDs in required format
    for doc in docs_output:
        f.write(f"{doc}\n")


FileNotFoundError: [Errno 2] No such file or directory: '/Users/niyaznabi/Desktop/IR Project/stopwordlist.txt'